# PythonSparks eCommerce ETL Notebook

This Databricks-style notebook shows the same steps as the local PySpark pipeline: ingestion, cleaning, transformation, and analytics.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, concat_ws, to_date, lower, expr

spark = SparkSession.builder.appName('PythonSparksNotebook').getOrCreate()

In [ ]:
sales_path = '/dbfs/FileStore/tables/sales.csv'
customers_path = '/dbfs/FileStore/tables/customers.json'
products_path = '/dbfs/FileStore/tables/products.csv'
orders_path = '/dbfs/FileStore/tables/orders.json'

sales_df = spark.read.option('header', 'true').option('inferSchema', 'true').csv(sales_path)
customers_df = spark.read.option('multiline', 'false').json(customers_path)
products_df = spark.read.option('header', 'true').option('inferSchema', 'true').csv(products_path)
orders_df = spark.read.option('multiline', 'false').json(orders_path)

display(sales_df.limit(5))
display(customers_df.limit(5))

In [ ]:
sales_clean = sales_df
    .withColumn('order_id', col('order_id').cast('integer'))
    .withColumn('customer_id', col('customer_id').cast('integer'))
    .withColumn('product_id', col('product_id').cast('integer'))
    .withColumn('quantity', col('quantity').cast('integer'))
    .withColumn('price', col('price').cast('double'))
    .withColumn('order_date', to_date(col('order_date'), 'yyyy-MM-dd'))
    .withColumn('sales_amount', col('quantity') * col('price'))
    .filter(col('order_id').isNotNull())
    .filter(col('quantity') > 0)
    .filter(col('price') >= 0)

customers_clean = customers_df
    .withColumn('customer_id', col('customer_id').cast('integer'))
    .withColumn('signup_date', to_date(col('signup_date'), 'yyyy-MM-dd'))
    .withColumn('full_name', concat_ws(' ', col('first_name'), col('last_name')))
    .withColumn('email', lower(col('email')))
    .dropDuplicates(['customer_id'])

display(sales_clean.limit(5))
display(customers_clean.limit(5))

## Build the fact table and run SQL analysis

In [ ]:
orders_clean = orders_df
    .withColumn('order_id', col('order_id').cast('integer'))
    .withColumn('shipping_city', lower(col('shipping_city')))
    .withColumn('shipping_state', lower(col('shipping_state')))
    .withColumn('payment_type', lower(col('payment_type')))
    .dropDuplicates(['order_id'])

products_clean = products_df
    .withColumn('product_id', col('product_id').cast('integer'))
    .withColumn('price', col('price').cast('double'))
    .withColumn('cost', col('cost').cast('double'))
    .dropDuplicates(['product_id'])

fact_orders = sales_clean
    .join(customers_clean, 'customer_id', 'left')
    .join(products_clean, 'product_id', 'left')
    .join(orders_clean, 'order_id', 'left')
    .withColumn('gross_margin', col('sales_amount') - (col('quantity') * col('cost')))
    .withColumn('validation_status', expr("CASE WHEN full_name IS NULL OR product_name IS NULL THEN 'MISSING_DIMENSION' ELSE 'OK' END"))

fact_orders.createOrReplaceTempView('fact_orders')

display(spark.sql('SELECT category, SUM(sales_amount) AS revenue FROM fact_orders GROUP BY category ORDER BY revenue DESC'))